### Build a basic chatbot with LangChain

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

### Invoking LLM

In [5]:
load_dotenv(override=True)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7, max_tokens=1000, max_retries=3, request_timeout=30)
messages = [
    SystemMessage(content="You are a helpful Java developer."),
    HumanMessage(content="Write a Java function that takes a string as input and returns the string reversed."),
]

response = llm.invoke(messages)
print(response.content)

Certainly! Here's a simple Java function that takes a string as input and returns the reversed string:

```java
public class StringUtil {
    public static String reverseString(String input) {
        if (input == null) {
            return null; // or throw an exception if preferred
        }
        return new StringBuilder(input).reverse().toString();
    }

    public static void main(String[] args) {
        String original = "Hello, World!";
        String reversed = reverseString(original);
        System.out.println("Original: " + original);
        System.out.println("Reversed: " + reversed);
    }
}
```

This function uses `StringBuilder`'s built-in `reverse()` method for simplicity and efficiency.


#### 1.Data Ingestion

Text File loader

In [6]:
### Text file loader
from langchain_community.document_loaders import TextLoader

loader = TextLoader('sample.txt')
load_doc = loader.load()
load_doc

[Document(metadata={'source': 'sample.txt'}, page_content='This is some text! Q2 Baby.')]

PDF file Loader

In [14]:
## pdf file loader
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

pdf_path = Path('../Attention_is_all_you_need.pdf')
print(f"PDF file path: {pdf_path.resolve()}")

loader = PyPDFLoader(pdf_path)
load_doc = loader.load()
load_doc[0].page_content[:500]  # Display the first 500 characters of the first page

PDF file path: C:\Users\Vidhan Chandra\Downloads\ML AI Projects\LangGraphCourse\Attention_is_all_you_need.pdf


'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or'

Web Based loader

In [ ]:
## Web Based loader
from langchain_community.document_loaders import WebBaseLoader
from bs4 import SoupStrainer

# Extract only heading and paragraph tags
bs4_strainer = SoupStrainer(
    ["h1", "h2", "h3", "p"]
)

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
web_loader = WebBaseLoader(web_paths=(url,), bs_kwargs={
    "parse_only": bs4_strainer
})
web_doc = web_loader.load()
web_doc

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence'}, page_content='ContentsArtificial intelligenceArtificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing,

##### CSV Loader

In [17]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path='employees.csv')
docs = loader.load()
print(docs[0].page_content)  # Display the content of the first row

id: 1
name: Alice
department: Engineering
salary: 80000


### 2.Data Transformation

##### Text Splitting from Documents 

In [12]:
## pdf file loader
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

pdf_path = Path('../Attention_is_all_you_need.pdf')
print(f"PDF file path: {pdf_path.resolve()}")

loader = PyPDFLoader(pdf_path)
load_doc = loader.load()
load_doc

PDF file path: C:\Users\Vidhan Chandra\Downloads\ML AI Projects\LangGraphCourse\Attention_is_all_you_need.pdf


[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
text_splitter.create_documents([load_doc[0].page_content])  # Split the content of the first page into chunks

[Document(metadata={}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe superior in quality while being more paral